In [0]:
# ============================================================
# 03_evaluation_harness — Stage 5
# PROBE: can we call a live Agent Bricks agent from notebook code?
#   If yes  -> Option 2 (agent-level evaluation) is possible.
#   If no   -> Option 1 (tool/gold-layer evaluation), documented as a finding.
# ============================================================

# Step 1: find the serving endpoint name for our agents
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

print("Serving endpoints visible to this notebook:")
try:
    for ep in w.serving_endpoints.list():
        print(f"  - {ep.name}   (state: {ep.state.ready if ep.state else 'n/a'})")
except Exception as e:
    print("  ERROR listing endpoints:", e)

In [0]:
# ============================================================
# Stage 5 — agent invocation helper
#   Calls a live agent endpoint, returns a clean structured result:
#   final answer text, the tool(s) called, and the arguments.
# ============================================================
import requests, json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
HOST  = ctx.apiUrl().get()
TOKEN = ctx.apiToken().get()

# friendly names -> endpoint names (confirm the mapping is right!)
AGENT_ENDPOINTS = {
    "cash_flow":   "mas-9cbc4569-endpoint",
    "vendor":      "mas-4a9cde78-endpoint",
    "supervisor":  "mas-43f19b70-endpoint",
}

def ask_agent(endpoint_name, question, timeout=120):
    """Call an agent endpoint; return a structured dict."""
    url = f"{HOST}/serving-endpoints/{endpoint_name}/invocations"
    r = requests.post(
        url,
        headers={"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"},
        json={"input": [{"role": "user", "content": question}]},
        timeout=timeout,
    )
    if r.status_code != 200:
        return {"ok": False, "error": f"HTTP {r.status_code}: {r.text[:300]}",
                "answer": "", "tools_called": [], "tool_args": []}

    body = r.output if hasattr(r, "output") else r.json()
    outputs = body.get("output", [])

    # final answer = last message-type item's text
    answer = ""
    for item in outputs:
        if item.get("type") == "message":
            for c in item.get("content", []):
                if c.get("type") == "output_text":
                    answer = c.get("text", "")   # keep overwriting -> ends on last

    # tools called + their arguments
    tools_called = [i.get("name") for i in outputs if i.get("type") == "function_call"]
    tool_args    = [i.get("arguments") for i in outputs if i.get("type") == "function_call"]

    return {"ok": True, "error": None, "answer": answer,
            "tools_called": tools_called, "tool_args": tool_args}

# --- quick self-test on all three agents ---
for label, ep in AGENT_ENDPOINTS.items():
    print("=" * 60)
    print(f"AGENT: {label}  ({ep})")
    res = ask_agent(ep, "What was the total net cash flow for company code 1010?")
    print("  ok:", res["ok"])
    print("  tools_called:", res["tools_called"])
    print("  answer:", res["answer"][:200])

In [0]:
# ============================================================
# Stage 5 — corrected endpoint mapping + proper self-test
# ============================================================
AGENT_ENDPOINTS = {
    "cash_flow":   "mas-9cbc4569-endpoint",
    "vendor":      "mas-43f19b70-endpoint",
    "supervisor":  "mas-4a9cde78-endpoint",
}

# domain-appropriate self-test for each agent
tests = {
    "cash_flow":  "What was the total net cash flow for company code 1010?",
    "vendor":     "How is vendor V008 performing?",
    "supervisor": "Which vendors have the worst OTIF?",
}

for label, q in tests.items():
    ep = AGENT_ENDPOINTS[label]
    print("=" * 60)
    print(f"AGENT: {label}  ({ep})")
    print(f"Q: {q}")
    res = ask_agent(ep, q)
    print("  ok:", res["ok"])
    print("  tools_called:", res["tools_called"])
    print("  answer:", res["answer"][:220])

In [0]:
# ============================================================
# Stage 5 / Part 1 — the GOLDEN TEST SET
#   Representative test cases with expected behaviour.
#   Stored as a Delta table so it is governed and versioned.
# ============================================================
from pyspark.sql import Row

# Each case: which agent, the question, the tool we expect it to call,
# a string the answer MUST contain, and a note on what it verifies.
golden_cases = [
    # --- cash flow agent ---
    ("G01", "cash_flow", "What was the total net cash flow for company code 1010?",
     "get_cashflow_summary", "16.63", "Single-company cash flow lookup + correct figure"),
    ("G02", "cash_flow", "Which 5 company codes have the highest net cash flow?",
     "rank_companies_by_cashflow", "1710", "Cash flow ranking, top result correct"),

    # --- vendor agent ---
    ("G03", "vendor", "How is vendor V008 performing?",
     "get_vendor_performance", "97", "Single-vendor lookup + correct OTIF figure"),
    ("G04", "vendor", "Which 5 vendors have the worst OTIF?",
     "rank_vendors_by_metric", "V004", "Vendor ranking, worst performer correct"),
    ("G05", "vendor", "Tell me about vendor V017",
     "get_vendor_performance", "caution", "GOVERNANCE: low-confidence warning must appear"),

    # --- supervisor (multi-agent routing) ---
    ("G06", "supervisor", "What was the total net cash flow for company code 1010?",
     "get_cashflow_summary", "16.63", "Supervisor routes a cash question to cash sub-agent"),
    ("G07", "supervisor", "Which vendors have the worst OTIF?",
     "rank_vendors_by_metric", "V004", "Supervisor routes a vendor question to vendor sub-agent"),
    ("G08", "supervisor", "Tell me about vendor V017",
     "get_vendor_performance", "caution", "GOVERNANCE survives the hierarchy: warning still appears"),
]

schema = "case_id STRING, agent STRING, question STRING, expect_tool STRING, expect_contains STRING, verifies STRING"
golden_df = spark.createDataFrame(
    [Row(case_id=c[0], agent=c[1], question=c[2], expect_tool=c[3],
         expect_contains=c[4], verifies=c[5]) for c in golden_cases],
    schema=schema,
)

golden_df.write.mode("overwrite").saveAsTable("workspace.default.eval_golden_set")

print(f"Golden test set saved: {golden_df.count()} cases -> workspace.default.eval_golden_set")
golden_df.select("case_id", "agent", "question", "expect_tool", "verifies").show(truncate=False)

In [0]:
# ============================================================
# Stage 5 / Part 2 — the SCORERS
#   Code-based scorers (the honest equivalent of LLM judges).
#   Each scores one quality dimension of an agent's response.
# ============================================================

def score_routing(result, expect_tool):
    """Did the agent call the expected tool? (routing / tool-selection quality)
       Note: tools_called may include sub-agent ids (sa-...) for the supervisor;
       we check the expected tool appears anywhere in the chain."""
    called = result.get("tools_called", [])
    hit = any(expect_tool in (t or "") for t in called)
    return {
        "dimension": "routing",
        "pass": hit,
        "detail": f"expected '{expect_tool}' in {called}",
    }

LOW_DATA_SIGNALS = ["1 purchase order", "one purchase order", "few orders",
                    "limited data", "limited order", "single order", "1 order",
                    "one order", "insufficient", "only 1", "1 po"]
CAUTION_SIGNALS  = ["caution", "not statistically reliable", "not reliable",
                    "unreliable", "not representative", "treat", "cannot be",
                    "should not", "limited", "too few"]

def score_correctness(result, expect_contains):
    """For governance cases, pass if the answer signals BOTH that the vendor
       has little data AND that the figures are unreliable. Otherwise the
       answer must contain the literal expected string."""
    answer = (result.get("answer") or "").lower()
    if expect_contains == "GOVERNANCE":
        has_low_data = any(s in answer for s in LOW_DATA_SIGNALS)
        has_caution  = any(s in answer for s in CAUTION_SIGNALS)
        ok = has_low_data and has_caution
        return {"dimension": "correctness", "pass": ok,
                "detail": f"governance: low_data={has_low_data}, caution={has_caution}"}
    ok = expect_contains.lower() in answer
    return {"dimension": "correctness", "pass": ok,
            "detail": f"expected answer to contain '{expect_contains}'"}

def score_responded(result):
    """Did the agent respond at all, with a non-empty answer? (availability)"""
    ok = result.get("ok", False) and bool((result.get("answer") or "").strip())
    return {
        "dimension": "responded",
        "pass": ok,
        "detail": result.get("error") or "non-empty answer returned",
    }

def score_case(golden_case, result):
    """Run all scorers for one golden case. Returns overall pass + per-dimension."""
    scores = [
        score_responded(result),
        score_routing(result, golden_case["expect_tool"]),
        score_correctness(result, golden_case["expect_contains"]),
    ]
    overall = all(s["pass"] for s in scores)
    return {"overall_pass": overall, "scores": scores}

# --- self-test the scorers on one live call ---
test_case = {"expect_tool": "get_cashflow_summary", "expect_contains": "16.63"}
test_result = ask_agent(AGENT_ENDPOINTS["cash_flow"],
                        "What was the total net cash flow for company code 1010?")
demo = score_case(test_case, test_result)

print("Scorer self-test on G01-style case:")
print("  overall_pass:", demo["overall_pass"])
for s in demo["scores"]:
    flag = "PASS" if s["pass"] else "FAIL"
    print(f"  [{flag}] {s['dimension']:12} - {s['detail']}")

In [0]:
# ============================================================
# Stage 5 / Part 3 — the HARNESS RUNNER
#   Runs every golden case against the live agents, scores it,
#   and appends results (with run timestamp) to a Delta table.
#   This results table is the drift-detection record.
# ============================================================
from pyspark.sql import Row
from datetime import datetime
import uuid

def run_evaluation():
    run_id  = str(uuid.uuid4())[:8]
    run_ts  = datetime.now()
    print(f"=== Evaluation run {run_id}  @ {run_ts:%Y-%m-%d %H:%M:%S} ===\n")

    golden = spark.table("workspace.default.eval_golden_set").collect()
    rows = []

    for g in golden:
        gc = g.asDict()
        endpoint = AGENT_ENDPOINTS[gc["agent"]]
        result   = ask_agent(endpoint, gc["question"])
        scored   = score_case(gc, result)

        per_dim = {s["dimension"]: s["pass"] for s in scored["scores"]}
        flag = "PASS" if scored["overall_pass"] else "FAIL"
        print(f"[{flag}] {gc['case_id']} ({gc['agent']:10}) "
              f"responded={per_dim['responded']} routing={per_dim['routing']} "
              f"correctness={per_dim['correctness']}")
        if not scored["overall_pass"]:
            print(f"        verifies: {gc['verifies']}")
            print(f"        answer:   {(result.get('answer') or '')[:160]}")

        rows.append(Row(
            run_id=run_id, run_ts=run_ts, case_id=gc["case_id"],
            agent=gc["agent"], verifies=gc["verifies"],
            responded=per_dim["responded"], routing=per_dim["routing"],
            correctness=per_dim["correctness"], overall_pass=scored["overall_pass"],
            answer_snippet=(result.get("answer") or "")[:300],
        ))

    results_df = spark.createDataFrame(rows)
    results_df.write.mode("append").saveAsTable("workspace.default.eval_results")

    total  = len(rows)
    passed = sum(1 for r in rows if r["overall_pass"])
    print(f"\n=== Run {run_id} complete: {passed}/{total} passed "
          f"({100*passed/total:.0f}%) — appended to workspace.default.eval_results ===")
    return run_id, passed, total

# run it now — the first baseline evaluation
run_evaluation()

In [0]:
# ============================================================
# Stage 5 / Part 3b — tune the governance scorer
#   FINDING from the baseline run: G08 "failed" but the agent was
#   correct - it warned "too few orders for reliable metrics"
#   instead of the literal word "caution". The scorer was too
#   brittle. Fix: the governance check accepts ANY of several
#   equivalent warning phrasings. This is scorer calibration -
#   the "who judges the judge" discipline.
# ============================================================

# governance cases: pass if the answer contains ANY of these phrases
GOVERNANCE_PHRASES = ["caution", "too few orders", "not statistically reliable",
                      "not reliable", "unreliable", "limited order", "insufficient"]

def score_correctness(result, expect_contains):
    """Correctness scorer. For governance cases (expect_contains == 'GOVERNANCE'),
       pass if ANY recognised low-confidence warning phrase appears.
       Otherwise, the answer must contain the literal expected string."""
    answer = (result.get("answer") or "").lower()
    if expect_contains == "GOVERNANCE":
        hit = any(p in answer for p in GOVERNANCE_PHRASES)
        return {"dimension": "correctness", "pass": hit,
                "detail": f"governance warning present (any of {GOVERNANCE_PHRASES})"}
    hit = expect_contains.lower() in answer
    return {"dimension": "correctness", "pass": hit,
            "detail": f"expected answer to contain '{expect_contains}'"}

# update G05 and G08 in the golden set to use the GOVERNANCE marker
spark.sql("""
UPDATE workspace.default.eval_golden_set
SET expect_contains = 'GOVERNANCE'
WHERE case_id IN ('G05', 'G08')
""")

print("Scorer tuned. G05 and G08 now use the GOVERNANCE marker.")
spark.table("workspace.default.eval_golden_set").select(
    "case_id", "agent", "expect_tool", "expect_contains", "verifies"
).orderBy("case_id").show(truncate=False)

In [0]:
run_evaluation()